# Evaluating and Selecting Supervised Models

You can now fit several different models to the same problem. That raises a question this course
has so far dodged: **how do you decide which one to keep?**

The obvious answer — try them and pick the one that scores best — is wrong in a way that is easy to
miss and expensive to discover late. This session is about doing it correctly.

## Learning objectives

By the end of this session you will be able to:

1. Explain why a score measured on the training data cannot be used to choose a model, and
   demonstrate it on real data.
2. Describe the distinct roles of the **training**, **validation** and **test** sets, and say what
   is spent when each one is used.
3. Explain **k-fold cross-validation**, and use it to compare candidate models.
4. Read a cross-validation result as a **range rather than a point**, and judge when a difference
   between two models is too small to act on.
5. Describe the **bias–variance tradeoff**, and recognize which side of it a model sits on from its
   training and validation errors.
6. Run a complete model-selection workflow that keeps the test set honest.

## 1. Why a single split is not enough

The basic evaluation recipe you already know is:

1. Split the data into a training set and a test set.
2. Fit a model on the training set.
3. Score it on the test set.

This is sound for measuring *one* model. It breaks the moment you have several and want to choose.

The reason is that **choosing is itself a decision made from data.** If you fit three models, look
at their test scores and keep the best, then the test set has participated in building your final
answer — through you rather than through `fit`, but it has participated. The score you go on to
report is no longer an estimate of performance on unseen data, because that data is not unseen any
more: you used it to pick a winner. Report it anyway and you will be optimistic, sometimes badly.

The fix is to introduce a third set, so that choosing and reporting use different data.

| Set | What it is used for | How often you may use it |
|---|---|---|
| **Training** | fitting each candidate's parameters | as often as you like |
| **Validation** | comparing candidates and choosing one | as often as you like |
| **Test** | reporting the performance of the chosen model | **once, at the very end** |

The discipline is that the test set is spent on its first use. Everything exploratory — comparing
models, tuning settings, deciding whether a feature helps — happens against the validation set.

### The problem with a single validation set

Holding out a validation set keeps the test set clean, but it introduces two problems of its own.

**The estimate is unreliable.** Which rows land in the validation set is decided by a shuffle, and
the set is necessarily small — it is a slice taken out of data you already did not have enough of. A
handful of awkward cases moving from one side of the split to the other can then change a model's
score by more than the real difference in quality between two good models. Taking the highest score
identifies the model that suited that particular draw, not the better model. Reshuffle, and a
different one may win.

**It costs training data.** Every row in the validation set is a row no model learns from, so each
candidate is a little worse than it could have been.

The two pull in opposite directions: a bigger validation set steadies the estimate but weakens the
models, a smaller one does the reverse. No size fixes both.

Cross-validation sidesteps the choice — it uses every row for validation and every row for training,
just never at the same time.

## 2. Cross-validation

**Cross-validation** is a procedure for estimating how well a model generalizes, by validating it
several times on different subsets of the data and combining the results.

The most common form is **k-fold cross-validation**, and the idea is small:

1. Split the training data into $k$ equal parts, called **folds**.
2. Take fold 1 as the validation set, fit the model on the remaining $k-1$ folds, and score it.
3. Repeat, each time using a different fold for validation.
4. You now have $k$ scores. Report their **mean** and their **spread**.

With $k = 5$ the pattern looks like this — each row is one round of fitting and scoring:

| Round | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|---|---|---|---|---|---|
| 1 | **validate** | train | train | train | train |
| 2 | train | **validate** | train | train | train |
| 3 | train | train | **validate** | train | train |
| 4 | train | train | train | **validate** | train |
| 5 | train | train | train | train | **validate** |

Every row is used for validation exactly once and for training $k-1$ times, so no data is wasted
and no single unlucky split decides the outcome. Typical choices are $k = 5$ or $k = 10$.

**What it costs.** You fit the model $k$ times instead of once, so it is $k$ times slower. And it
does not hand you a model to deploy — it evaluates a *recipe*, not a fitted object. Once
cross-validation tells you which recipe is best, you refit that one on the whole training set.

**What the spread tells you.** The mean says how good the model is. The standard deviation across
folds says how much that answer depends on which data it was trained on — a quantity we will use
directly in section 7.

> **In `scikit-learn`.** `cross_val_score(model, X, y, cv=5, scoring=...)` performs the whole
> procedure and returns the $k$ scores as an array. It clones the model for each fold, so the object
> you pass in is left unfitted. One trap: `scikit-learn` scorers always follow the rule *higher is
> better*, so error metrics are negated — asking for `"neg_root_mean_squared_error"` returns
> negative numbers whose magnitude is the RMSE. Multiply by $-1$ before reading them.

## 3. The data

We will use the **Auto MPG** dataset, a standard benchmark assembled from 1970s and 1980s car
models and distributed with this repository at `data/auto_mpg.csv`.

- **392 rows and 7 columns.**
- **One row is one car model.**
- **The label is `mpg`** — fuel efficiency in miles per gallon, running from 9.0 to 46.6. Predicting
  it is a **regression** problem.
- The features describe the car:
  - `displ` — engine displacement (cubic inches), 68 to 455
  - `hp` — horsepower, 46 to 230
  - `weight` — weight in pounds, 1,613 to 5,140
  - `accel` — seconds taken to accelerate from 0 to 60 mph, 8.0 to 24.8
  - `size` — a coarse size class taking one of five values (7.5, 10, 12.5, 15, 20)
  - `origin` — where the model came from: `US` (245 cars), `Asia` (79) or `Europe` (68)
- **There are no missing values.**

It suits this session for two reasons. It is small enough that cross-validation runs instantly, and
its relationships are strong but not trivial — heavier, more powerful cars use more fuel, but not
linearly — so different model families genuinely disagree about it, which is what makes choosing
between them a real decision rather than a formality.

The cell below imports everything the notebook needs. `KNeighborsRegressor` is the one model here
you have not met; section 5 introduces it.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42

The cell below loads the file and confirms its shape and that nothing is missing.

In [ ]:
data = pd.read_csv("data/auto_mpg.csv")

print(f"{data.shape[0]} rows x {data.shape[1]} columns")
print(f"missing values: {data.isna().sum().sum()}")

Numbers do not show you what a car looks like in this table. The cell below prints the first five
rows — check each column against the description above, and note that `origin` is text while
everything else is numeric.

In [ ]:
data.head()

The cell below summarises the numeric columns. The point to notice is the **scale difference**:
`weight` averages around 2,978 while `accel` averages around 15.5. That will matter in section 5.

In [ ]:
data.describe().round(2)

## 4. Splitting and preparing the data

We separate the label from the features and hold out a test set immediately — before looking at
any model. Note that we do **not** carve out a separate validation set: cross-validation will
manufacture validation sets from the training data as it goes, which is the whole point of
section 2.

The cell below performs the split and reports the sizes.

In [ ]:
X = data.drop(columns="mpg")
y = data["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"training rows: {X_train.shape[0]}")
print(f"test rows:     {X_test.shape[0]}")

`origin` is text, and no model here can consume text. We convert it into numeric columns with
one-hot encoding, leaving the numeric features untouched.

> **In `scikit-learn`.** `ColumnTransformer` applies different transformations to different columns
> and `remainder="passthrough"` carries the rest through unchanged. `drop="first"` omits one
> category — with three origins, two columns are enough to identify all three, and dropping one
> avoids a redundancy that upsets linear models. `set_output(transform="pandas")` keeps the result a
> DataFrame instead of a bare array, so the column names survive. The encoder is fitted on the
> training data **only**, then applied to the test data, so nothing about the test set leaks into
> the transformation.

The cell below builds the encoder, fits it on the training features and applies it to both halves,
then shows the result.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(sparse_output=False, drop="first"), ["origin"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train_preproc = preprocessor.fit_transform(X_train)
X_test_preproc = preprocessor.transform(X_test)

X_train_preproc.head()

## 5. The candidates

We will compare four recipes. Three you have met; one is new.

**The baseline.** A model that ignores the features and always predicts the mean `mpg` of the
training cars. It exists so that every other number has something to be compared against.

**Linear regression.** Assumes `mpg` is a weighted sum of the features plus a constant.

**A decision tree** of depth 4. A short cascade of yes/no questions on the features.

**k-nearest neighbours**, which is new, and the idea is almost embarrassingly simple. To predict the
`mpg` of a car, find the $k$ cars in the training data that are most *similar* to it, and average
their `mpg`. That is the whole algorithm. Two consequences are worth stating now:

- **Nothing is really "fitted".** There are no weights to solve for and no tree to grow — the
  training data *is* the model, and `fit` mostly just stores it. The work happens at prediction
  time, when the neighbours have to be found.
- **"Similar" means close in distance**, computed across all the feature columns at once. That
  makes the algorithm sensitive to the *scale* of those columns: a feature measured in thousands
  will dominate the distance and a feature measured in single digits will barely register. We
  deliberately leave that unaddressed for now and return to it in section 8.

> **In `scikit-learn`.** `KNeighborsRegressor(n_neighbors=5)` is the default; `n_neighbors` is $k$.

The cell below collects all four into a dictionary, so that everything afterwards can loop over
them instead of repeating itself four times.

In [ ]:
models = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "KNN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "Linear regression": LinearRegression(),
    "Decision tree (depth 4)": DecisionTreeRegressor(
        max_depth=4, random_state=RANDOM_STATE
    ),
}

## 6. Measuring the error

We need one number per model. For regression, a standard choice is the **root mean squared error**:

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

where:

- $n$ — the number of cars being scored.
- $y_i$ — the true `mpg` of car $i$.
- $\hat{y}_i$ — the `mpg` the model predicted for car $i$.
- $\text{RMSE}$ — runs from $0$ (perfect predictions) upward, with no fixed maximum. It is measured
  in the units of the label, so an RMSE of $4.2$ means a typical miss of about 4.2 miles per gallon.

Squaring the errors before averaging makes large mistakes count disproportionately, and the square
root at the end returns the result to the label's own units so it can be interpreted directly.
Unlike $R^2$, **lower is better**.

The cell below fits every candidate on the training data and records its RMSE on that same training
data. Watch which model comes out on top.

In [ ]:
results = []
for name, model in models.items():
    model.fit(X_train_preproc, y_train)
    train_rmse = root_mean_squared_error(y_train, model.predict(X_train_preproc))
    results.append({"model": name, "train_rmse": train_rmse})

results = pd.DataFrame(results)
results.round(3)

Every real model beats the baseline's RMSE of about 7.74, so all three have learned something.

And the **decision tree wins**, at about 3.13 against KNN's 3.44 and linear regression's 4.08.

If training error were a fair basis for choosing, we would stop here and ship the tree. Section 1
argued that it is not. The next section shows what that argument is worth in miles per gallon.

## 7. Choosing with cross-validation

Now we score the same four recipes properly: five-fold cross-validation on the training data, so
that every score is measured on cars the model did not learn from.

The cell below runs `cross_val_score` for each candidate, flips the sign of the negated scores, and
records the mean and standard deviation of the five folds alongside the training error we already
have.

In [ ]:
cv_means, cv_stds = [], []
for name, model in models.items():
    fold_scores = -cross_val_score(
        model,
        X_train_preproc,
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    cv_means.append(fold_scores.mean())
    cv_stds.append(fold_scores.std())

results["cv_rmse_mean"] = cv_means
results["cv_rmse_std"] = cv_stds
results.round(3)

The ranking turns over.

| Model | train RMSE | CV RMSE | CV std |
|---|---|---|---|
| Baseline (mean) | 7.742 | 7.753 | 0.485 |
| KNN (k=5) | 3.443 | 4.414 | 0.233 |
| Linear regression | 4.080 | **4.180** | 0.281 |
| Decision tree (depth 4) | **3.125** | 4.411 | 0.399 |

**The decision tree had the best training error and has the worst cross-validated error.** It looked
like the strongest model and it is the weakest of the three. Linear regression, which came last on
the training data, comes first here. Choosing on training error would have picked exactly the wrong
model — which is the entire argument of section 1, now with a number attached.

**But read the spread before declaring a winner.** Linear regression's cross-validated RMSE is
4.180 and KNN's is 4.414, a difference of 0.234. The standard deviations across folds are 0.281 and
0.233 — *the gap between the two models is smaller than the fold-to-fold noise within each of
them.* Cross-validation is telling us something more careful than "linear regression is best":

- It rules the decision tree **out** with reasonable confidence.
- It **cannot separate** KNN from linear regression on this data.

A cross-validation result is a range, not a point. Treating a difference of 0.23 as a real finding
when the measurement wobbles by 0.28 is how people convince themselves they have improved something
they have not. When two candidates are this close, choose on other grounds — interpretability,
prediction speed, how much you trust the assumptions — and say plainly that the data did not decide.

## 8. Bias and variance

The table above contains a second story, and it explains *why* the tree failed.

Every model's error comes from two competing sources.

**Bias** is error from the model being too rigid to represent the real pattern. A model with high
bias makes the same kind of mistake everywhere, and cannot fix it by being shown more data. Its
symptom is **high error even on the training set** — this is underfitting.

**Variance** is error from the model being too sensitive to the particular data it was trained on.
A model with high variance chases detail that happens to be in this training sample and will not be
in the next one. Its symptoms are **a large gap between training error and validation error**, and
**unstable scores across folds** — this is overfitting.

The tradeoff is that these pull in opposite directions. Making a model more flexible lowers bias
and raises variance; making it more rigid does the reverse. The best model is not the one that
minimizes either, but the one that minimizes their combined effect — which is precisely what a
cross-validated score measures.

Both symptoms are already in front of us. The cell below adds the gap between cross-validated and
training error, which is the quantity to read.

In [ ]:
results["cv_minus_train"] = results["cv_rmse_mean"] - results["train_rmse"]
results.round(3)

Read the last two columns together.

- **The baseline** has a gap of about 0.01 and a huge error everywhere. It is pure bias: so rigid
  that seeing different data changes nothing, and so rigid that it is useless.
- **Linear regression** has a gap of about 0.10. It is a stiff model — a single straight
  relationship — so it carries real bias, but almost no variance.
- **KNN** has a gap of about 0.97, and the **decision tree about 1.29**, the largest of the four.
  The tree fit the training cars closely and much of what it learned did not transfer. Its fold
  standard deviation, 0.399, is also the highest of the three real models: retrain it on slightly
  different data and you get a noticeably different answer. That is variance, visible twice.

### Watching the tradeoff move

The tree's depth controls exactly this. A shallower tree asks fewer questions and is more rigid
(more bias); a deeper one carves the training data more finely (more variance). The cell below
sweeps the depth and records training and cross-validated error at each setting.

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 12, None]
sweep = []
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
    tree.fit(X_train_preproc, y_train)
    fold_scores = -cross_val_score(
        tree, X_train_preproc, y_train, cv=5, scoring="neg_root_mean_squared_error"
    )
    sweep.append(
        {
            "max_depth": "none" if depth is None else str(depth),
            "train_rmse": root_mean_squared_error(y_train, tree.predict(X_train_preproc)),
            "cv_rmse": fold_scores.mean(),
        }
    )

sweep = pd.DataFrame(sweep)
sweep.round(3)

Training error falls all the way down — from about 4.95 at depth 1 to about 0.12 with no limit at
all, where the tree has essentially memorized every training car. Cross-validated error does not
follow it. It falls, reaches its lowest point at **depth 2**, and then climbs steadily.

That divergence is the bias–variance tradeoff, drawn by the data itself. The cell below plots both
curves so the shape is unmistakable; note that the deepest trees are read at the right-hand edge.

In [ ]:
positions = range(len(sweep))
best = sweep["cv_rmse"].idxmin()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(positions, sweep["train_rmse"], marker="o", label="training RMSE")
ax.plot(positions, sweep["cv_rmse"], marker="s", label="cross-validated RMSE")
ax.axvline(best, color="grey", linestyle="--", linewidth=1)
ax.annotate(
    f"lowest CV error\n(max_depth = {sweep.loc[best, 'max_depth']})",
    xy=(best, sweep.loc[best, "cv_rmse"]),
    xytext=(best + 0.5, sweep["cv_rmse"].max() - 0.35),
    arrowprops={"arrowstyle": "->", "color": "grey"},
    fontsize=9,
)
ax.set_xticks(list(positions))
ax.set_xticklabels(sweep["max_depth"])
ax.set_xlabel("max_depth  (more flexible to the right)")
ax.set_ylabel("RMSE (miles per gallon)")
ax.set_title("Deeper trees always fit the training data better,\nand stop helping on new data")
ax.margins(y=0.12)
ax.legend(loc="center right")
fig.tight_layout()
plt.show()

The left-hand end is **underfitting**: both curves are high, and the model is too rigid to capture
what is there. The right-hand end is **overfitting**: the training curve is on the floor while the
cross-validation curve has risen well above its minimum. The gap between the two curves *is* the
variance, growing as the tree gets deeper.

The useful model sits at the bottom of the cross-validation curve — and note that the depth-4 tree
we have been comparing all along is already past it. Depth 2 scores about 4.24 against depth 4's
4.41, which would have put the tree level with the other candidates rather than behind them. We did
not tune it, and it lost partly for that reason. Choosing a model and choosing its settings are the
same problem, and session 09 takes up the second half of it properly.

## 9. A caveat about KNN and scale

Section 5 flagged that k-nearest neighbours measures distance across all features at once, and that
our features are on very different scales — `weight` in the thousands, `accel` in the tens. In a
distance calculation, `weight` therefore drowns out everything else: two cars 500 pounds apart are
"far apart" no matter how similar the rest of their specifications are.

Putting every feature on a comparable scale first should help. The cell below cross-validates KNN
again with standardization applied ahead of it, chained into a single object so the scaling is
refitted inside each fold rather than leaking across them.

In [ ]:
scaled_knn = Pipeline(
    [("scale", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=5))]
)
scaled_scores = -cross_val_score(
    scaled_knn, X_train_preproc, y_train, cv=5, scoring="neg_root_mean_squared_error"
)

print(f"KNN unscaled : CV RMSE {results.loc[1, 'cv_rmse_mean']:.3f}")
print(f"KNN scaled   : CV RMSE {scaled_scores.mean():.3f} +/- {scaled_scores.std():.3f}")

Scaling moves KNN from about 4.41 to about 3.99 — a bigger improvement than the entire gap between
the three models in section 7, and enough to make it the best candidate we have seen.

The lesson is not really about KNN. It is that **the thing you are selecting is not a model, it is a
whole pipeline** — preprocessing included. Comparing a scale-sensitive algorithm against
scale-indifferent ones on unscaled data does not tell you which algorithm is better; it tells you
which one tolerates your preprocessing choices. Whenever you compare models, check first whether
they are being asked the same question.

## 10. The final evaluation

Cross-validation has done its job: it ruled out the depth-4 tree and left linear regression and KNN
too close to separate. Suppose we take linear regression, on the grounds that it scored marginally
best and is by far the easiest of the three to explain to somebody else.

Only now do we touch the test set — once, to report how the chosen model performs on data that has
played no part in any decision.

The cell below refits linear regression on the full training set and scores it on the test set.

In [ ]:
final_model = LinearRegression()
final_model.fit(X_train_preproc, y_train)

final_rmse = root_mean_squared_error(y_test, final_model.predict(X_test_preproc))
print(f"chosen model: linear regression")
print(f"test RMSE: {final_rmse:.3f} miles per gallon")

About 4.30 miles per gallon, close to the 4.18 cross-validation predicted — which is the outcome
you hope for, and evidence that the procedure worked.

For teaching purposes only, the cell below scores the other candidates on the test set too, so we
can check whether cross-validation's verdict held up. **This is not something you would do in real
work**: the moment you look at several test scores, the test set has been spent on comparison and
can no longer be reported as a clean estimate.

In [ ]:
for name, model in models.items():
    test_rmse = root_mean_squared_error(y_test, model.predict(X_test_preproc))
    print(f"{name:<26} test RMSE {test_rmse:.3f}")

Cross-validation was right about what mattered. It said the decision tree was the weakest of the
three, and the test set agrees emphatically — about 4.90 against roughly 4.2 to 4.3 for the others.
It also said KNN and linear regression could not be separated, and the test set agrees with that
too: 4.21 against 4.30, a difference no larger than the fold-to-fold wobble we already measured.

Notice that KNN edges linear regression here, having come second in cross-validation. That is not a
failure of the method — it is the method being honest. Two models that cross-validation declared
indistinguishable turned out to be, and their order on any one sample is a coin toss.

## 11. Wrap-up

- A score measured on the **training set cannot be used to choose a model**. On this data it would
  have selected the decision tree, which was the worst of the three candidates on every honest
  measurement.
- **Training** data fits parameters, **validation** data chooses between candidates, and the
  **test** set is spent once to report the result. Mixing the last two is the most common way to
  produce a number that will not survive contact with production.
- **k-fold cross-validation** replaces a single arbitrary validation split with $k$ of them, using
  all the data for both purposes and giving a mean *and* a spread.
- Read that spread. A difference between two models that is **smaller than the fold-to-fold
  variation** is not a finding. Cross-validation is at least as valuable for the differences it
  refuses to confirm as for the ones it does.
- **Bias** shows up as high error everywhere; **variance** shows up as a wide gap between training
  and validation error and as unstable folds. The depth sweep drew both, and put the best tree at
  depth 2 rather than the depth 4 we had assumed.
- You are always selecting a **pipeline**, not an algorithm. Scaling alone improved KNN by more than
  the gap that separated all three models.

### What is next

Two questions this session opened and deliberately left open.

The depth sweep chose between eight settings of one parameter by hand. Real models have several
parameters with many plausible values each, and the combinations multiply quickly.
**Session 09** covers searching that space systematically instead of by eye.

And every model here was a single model. **Session 10** asks what happens when you stop choosing
one and combine them instead — which turns out to be one of the most reliable ways to improve on
exactly the kind of comparison we just ran.